# EVO test

In [1]:
import pandas as pd
import os

## Load data

In [2]:
base_path = "../../data/perphect-data"

couples_df = pd.read_csv(os.path.join(base_path, "public_data_set", "couples_df.csv"))
phages_df = pd.read_csv(os.path.join(base_path, "public_data_set", "phages_df.csv"))
bacteria_df = pd.read_csv(os.path.join(base_path, "public_data_set", "bacteria_df.csv"))
print('couples_df.shape = ', couples_df.shape)
print('phages_df.shape = ', phages_df.shape)
print('bacteria_df.shape = ', bacteria_df.shape)

couples_df.shape =  (4202, 4)
phages_df.shape =  (3459, 3)
bacteria_df.shape =  (94, 3)


In [3]:
phages_df.head()

,phage_id,phage_sequence,sequence_length
0,5326,TGCGGCCGCCCCATCCTGTACGGGTTTCCAAGTCGATCGGAGGGCA...,53124
1,6247,GGCTTTCGTGTGAGCCGTGATGTTTTCACGAATATGTGCCCCACCT...,74483
2,1976,GTGGGAATTTTTTTTTTGGGTTGCGCGGTGATCGCCGATGACGACG...,50781
3,430,ATGGCTTCGACTCAGACTCCAGCCGTCGGCAAGACCACGGCCATCG...,71565
4,431,TGCGGCTGAGCCATCGTGTACGGGTTTCCAAGTCCATCAGAGCCGG...,53396


## Embeddings

In [4]:
import torch
from evo import Evo
import gc

torch.__version__

'2.1.2+cu121'

In [33]:
device = 'cuda:3'
# device = 'cpu'

def clean_gpu():
    with torch.no_grad():
        torch.cuda.empty_cache()
        gc.collect()
clean_gpu()
device

'cuda:3'

In [6]:
evo_model = Evo('evo-1-8k-base', device=device)
model, tokenizer = evo_model.model, evo_model.tokenizer

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [7]:
model.eval()

StripedHyena(
  (embedding_layer): VocabParallelEmbedding(512, 4096)
  (norm): RMSNorm()
  (unembed): VocabParallelEmbedding(512, 4096)
  (blocks): ModuleList(
    (0-7): 8 x ParallelGatedConvBlock(
      (pre_norm): RMSNorm()
      (post_norm): RMSNorm()
      (filter): ParallelHyenaFilter()
      (projections): Linear(in_features=4096, out_features=12288, bias=True)
      (out_filter_dense): Linear(in_features=4096, out_features=4096, bias=True)
      (mlp): ParallelGatedMLP(
        (l1): Linear(in_features=4096, out_features=10928, bias=False)
        (l2): Linear(in_features=4096, out_features=10928, bias=False)
        (l3): Linear(in_features=10928, out_features=4096, bias=False)
      )
    )
    (8): AttentionBlock(
      (pre_norm): RMSNorm()
      (post_norm): RMSNorm()
      (inner_mha_cls): MHA(
        (rotary_emb): RotaryEmbedding()
        (Wqkv): Linear(in_features=4096, out_features=12288, bias=True)
        (inner_attn): FlashSelfAttention(
          (drop): Dropout(

In [8]:
# monkey patch the unembed function with identity
# this removes the final projection back from the embedding space into tokens
# so the "logits" of the model is now the final layer embedding
# see source for unembed - https://huggingface.co/togethercomputer/evo-1-131k-base/blob/main/model.py#L339

from torch import nn
import torch

class CustomEmbedding(nn.Module):
  def unembed(self, u):
    return u

model.unembed = CustomEmbedding()

# end custom code

In [31]:
# sequence = 'ACGT'
sequence = phages_df["phage_sequence"].iloc[0][:2**10]
clean_gpu()
input_ids = torch.tensor(
    tokenizer.tokenize(sequence),
    dtype=torch.int,
).to(device).unsqueeze(0)
len(sequence)

1024

In [32]:
embed, _ = model(input_ids) # (batch, length, embed dim)

print('Embed: ', embed)
print('Shape (batch, length, embed dim): ', embed.shape)

OutOfMemoryError: CUDA out of memory. Tried to allocate 256.00 MiB. GPU 3 has a total capacty of 44.34 GiB of which 126.81 MiB is free. Including non-PyTorch memory, this process has 44.21 GiB memory in use. Of the allocated memory 43.02 GiB is allocated by PyTorch, and 884.47 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [ ]:
# you can now use embedding for downstream classification tasks
# you probably want to aggregate over position dimension
mean_emb = embed.mean(dim=1)

## Embeddings with huggingface

In [3]:
import gc, torch

device = "cuda:3"

def clean_gpu():
    torch.cuda.empty_cache()
    gc.collect()
clean_gpu()
device

'cuda:3'

In [4]:
# Load model directly
from transformers import AutoModelForCausalLM, AutoTokenizer
model = AutoModelForCausalLM.from_pretrained("togethercomputer/evo-1-131k-base", trust_remote_code=True, torch_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained("togethercomputer/evo-1-131k-base", trust_remote_code=True)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [5]:
model.to(device)
model.eval()

StripedHyenaModelForCausalLM(
  (backbone): StripedHyena(
    (embedding_layer): VocabParallelEmbedding(512, 4096)
    (norm): RMSNorm()
    (unembed): VocabParallelEmbedding(512, 4096)
    (blocks): ModuleList(
      (0-7): 8 x ParallelGatedConvBlock(
        (pre_norm): RMSNorm()
        (post_norm): RMSNorm()
        (filter): ParallelHyenaFilter()
        (projections): Linear(in_features=4096, out_features=12288, bias=True)
        (out_filter_dense): Linear(in_features=4096, out_features=4096, bias=True)
        (mlp): ParallelGatedMLP(
          (l1): Linear(in_features=4096, out_features=10928, bias=False)
          (l2): Linear(in_features=4096, out_features=10928, bias=False)
          (l3): Linear(in_features=10928, out_features=4096, bias=False)
        )
      )
      (8): AttentionBlock(
        (pre_norm): RMSNorm()
        (post_norm): RMSNorm()
        (inner_mha_cls): MHA(
          (Wqkv): Linear(in_features=4096, out_features=12288, bias=True)
          (inner_attn)

In [ ]:
clean_gpu()

with torch.no_grad():
    # dna = "ACGTAGCATCGGATCTATCTATCGACACTTGGTTATCGATCTACGAGCATCTCGTTAGC"
    dna = phages_df["phage_sequence"].iloc[0][:2**10] # EVO model can only handle up to 1024 bp with 40GB of memory
    # dna = bacteria_df["bacterium_sequence"].iloc[0][:2**15] 
    inputs = tokenizer(dna, return_tensors = 'pt')["input_ids"].to(device)
    hidden_states = model(inputs)[0] # [1, sequence_length, 768]

# embedding with mean pooling
embedding_mean = torch.mean(hidden_states[0], dim=0)
print(embedding_mean.shape)

Initializing inference params...


OutOfMemoryError: CUDA out of memory. Tried to allocate 512.00 MiB. GPU 3 has a total capacty of 44.34 GiB of which 236.81 MiB is free. Including non-PyTorch memory, this process has 44.10 GiB memory in use. Of the allocated memory 43.56 GiB is allocated by PyTorch, and 219.76 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [8]:
embedding_mean

tensor([ -7.6562, -23.1250, -23.1250, -23.1250, -23.1250, -23.1250, -23.1250,
        -23.1250, -23.1250, -23.1250, -23.1250, -23.1250, -23.1250, -23.1250,
        -23.1250, -23.1250, -23.1250, -23.1250, -23.1250, -23.1250, -23.1250,
        -23.1250, -23.1250, -23.1250, -23.1250, -23.1250, -23.1250, -23.1250,
        -23.1250, -23.1250, -23.1250, -23.1250, -23.1250, -23.1250, -23.1250,
        -23.1250, -23.1250, -23.1250, -23.1250, -23.1250, -23.1250, -23.1250,
        -23.1250, -23.1250, -23.1250, -23.1250, -23.1250, -23.1250, -23.1250,
        -23.1250, -23.1250, -23.1250, -23.1250, -23.1250, -23.1250, -23.1250,
        -23.1250, -23.1250, -23.1250, -23.1250, -23.1250, -23.1250, -23.1250,
        -23.1250, -23.1250,  -0.5586, -22.3750,   0.1279, -22.5000, -23.1250,
        -23.1250,  -0.2520, -23.3750, -23.1250, -23.1250, -15.7500, -23.1250,
        -15.0000,  -9.6250, -23.1250, -23.1250, -23.1250, -14.6875, -16.1250,
         -0.3027, -23.1250, -21.3750, -15.2500, -22.3750, -14.00